In [1]:
#imports and libraries
!py -m pip install seaborn
import numpy as np
import pandas as pd
import joblib as jl
import sys
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor,RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    root_mean_squared_error
)
from matplotlib import pyplot as plt
import seaborn as sb

sys.path.append("..")


  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
train=pd.read_csv("../data/processed/train.csv")
val=pd.read_csv("../data/processed/val.csv")
test=pd.read_csv("../data/processed/test.csv")

xtrainScaled_smote=np.load("../data/processed/xtrain_scaled.npy")
ytrainSeverity_smote=np.load("../data/processed/ytrain_severity.npy")

FEATURES = [

    "district_encoded","T2M_mean", "RH2M_mean", "PRECTOTCORR_sum", "WS10M_mean",
    "T2M_lag1", "PRECTOTCORR_lag1", "PRECTOTCORR_lag2",
    "t1_cases", "t2_cases",
    "water_proxy",
    "month_sin", "month_cos",
    "week_sin", "week_cos"
]
TARGET = "cases"

xtrainScaled=train[FEATURES].values
xvalScaled=val[FEATURES].values
xtestScaled=test[FEATURES].values
ytrain=train[TARGET].values
yval=val[TARGET].values
ytest=test[TARGET].values

In [3]:
# helper function to evaluate the model accuracy for the regressors

def regressorEvaluation(model,xtest,ytest):

    predictions=model.predict(xtest)
    meanSquareError=mean_squared_error(ytest, predictions)
    r2=r2_score(ytest, predictions)
    rootMeanSquareError=root_mean_squared_error(ytest, predictions)
    meanAbsoluteError=mean_absolute_error(ytest, predictions)
    print("The Model Performance metrics are:")
    print(f"R2 Score: {r2}")
    print(f"Mean Squared Error: {meanSquareError}")
    print(f"Root Mean Squared Error: {rootMeanSquareError}")
    print(f"Mean Absolute Error: {meanAbsoluteError}")

    return {
        "R2_Score": r2,
        "MeanSquaredError": meanSquareError,
        "RootMeanSquaredError": rootMeanSquareError,
        "MeanAbsoluteError": meanAbsoluteError
    }

In [5]:
#Linear Regression as baseline model

linearRegression=LinearRegression()

linearRegression.fit(xtrainScaled,ytrain)

linearEvaluations=regressorEvaluation(linearRegression,xvalScaled,yval)

cvScore=cross_val_score(linearRegression,xtrainScaled,ytrain,cv=5,scoring='r2')

print(f"Cross Validation Average R2 Score: {cvScore.mean():.4f}")
print(f"Cross Validation R2 Scores Std: {cvScore.std():.4f}")

The Model Performance metrics are:
R2 Score: 0.0075353288386919015
Mean Squared Error: 1334.3380214890162
Root Mean Squared Error: 36.52859183556103
Mean Absolute Error: 17.90758615431165
Cross Validation Average R2 Score: 0.6921
Cross Validation R2 Scores Std: 0.2844
